# Salary & Experience — Análise de Dados e Regressão Linear com Python

## Pergunta central

**Qual é a relação entre anos de experiência e salário nesta amostra, e até que ponto uma regressão linear simples consegue prever o salário?**

A análise combina EDA e modelagem preditiva. A relação observada é uma **associação**, não uma evidência causal.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/raw/salary_data.csv")
df.head()

## 1. Diagnóstico da base

In [ ]:
print("Dimensões:", df.shape)
print("\nValores ausentes:")
print(df.isna().sum())
print("\nDuplicatas:", df.duplicated().sum())
df.describe().round(2)

## 2. Relação entre experiência e salário

In [ ]:
correlation = df["YearsExperience"].corr(df["Salary"])
print(f"Correlação: {correlation:.3f}")

df.plot(x="YearsExperience", y="Salary", kind="scatter", figsize=(9, 6), title="Salário x anos de experiência")
plt.show()

## 3. Treino e teste

In [ ]:
X = df[["YearsExperience"]]
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("Treino:", len(X_train))
print("Teste:", len(X_test))

## 4. Regressão linear

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Intercepto:", round(model.intercept_, 2))
print("Coeficiente:", round(model.coef_[0], 2))

O coeficiente indica que, **dentro da relação linear estimada nesta amostra**, um ano adicional de experiência está associado a aproximadamente 9,4 mil unidades monetárias adicionais no salário previsto. Isso não representa efeito causal.


## 5. Avaliação no teste

In [ ]:
pred = model.predict(X_test)

print("R²:", round(r2_score(y_test, pred), 3))
print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("RMSE:", round(mean_squared_error(y_test, pred) ** 0.5, 2))

## 6. Validação cruzada

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

r2_cv = cross_val_score(LinearRegression(), X, y, cv=cv, scoring="r2")
mae_cv = -cross_val_score(LinearRegression(), X, y, cv=cv, scoring="neg_mean_absolute_error")

print("R² médio:", round(r2_cv.mean(), 3))
print("MAE médio:", round(mae_cv.mean(), 2))

## 7. Resíduos

In [ ]:
residuals = y_test.to_numpy() - pred

plt.figure(figsize=(9, 5))
plt.scatter(pred, residuals)
plt.axhline(0)
plt.xlabel("Salário previsto")
plt.ylabel("Resíduo")
plt.title("Resíduos no conjunto de teste")
plt.show()

## 8. Principais achados

- A correlação entre experiência e salário é elevada na amostra (`r ≈ 0,978`).
- A regressão linear apresenta `R² ≈ 0,902` no conjunto de teste.
- O MAE no teste fica em torno de 6,3 mil unidades monetárias.
- Em validação cruzada de 5 partes, o `R²` médio fica próximo de 0,922.
- A amostra é pequena: apenas 30 observações.
- O dataset contém apenas experiência e salário; formação, cargo, setor, localização e empresa não estão disponíveis.
- O projeto é didático e não deve ser usado para estimar salários reais fora desta base.
